# Capacity and Reliability Benchmark

This notebook estimates the recording capacity required for target reconstruction accuracy and the empirical success probability under missing data. It is intended as an exploratory companion to the main algorithm benchmark.

Inputs: simulation parameters defined in the first experiment cell.
Outputs: `results/fig1_required_capacity.csv`, `results/fig2_success_probability.csv`, and `figures/capacity-reliability.pdf`.

Runtime note: the stochastic loops are intentionally small enough for iteration, but can be increased before final figure generation.


In [ ]:
# Shared notebook setup
# Resolve the project root whether the notebook is launched from the repository
# root or directly from the notebooks directory, then expose the local src/ package.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import triplet_lineage as tl


In [ ]:
import cassiopeia as cas
import cProfile
from collections import defaultdict
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import time
from tqdm.auto import tqdm
from scipy.interpolate import interp1d

In [ ]:
# =========================================================
# =========================================================
num_simulations_fig2 = 100
num_batches_fig1 = 10
depth = 8
num_states = 5
lamb = 0.5
q_dist = dict(zip(range(1, num_states + 1), [1 / num_states] * num_states))

k_cand_list = [10, 20, 50, 100, 150, 200]
p_miss_list = [0.0, 0.1, 0.2]
target_acc_list = np.linspace(0.6, 0.95, 8)

# =========================================================
# =========================================================
print("========== Generating Data for Figure 2 (Success Probability) ==========")
results_fig2 = []
acc_threshold = 0.9

for p_miss in p_miss_list:
    for k_cand in k_cand_list:
        success_count = 0
        for _ in tqdm(range(num_simulations_fig2), desc=f"k={k_cand}, p_miss={p_miss}"):
            ground_truth_tree = tl.complete_binary_tree_sim(k_cand, q_dist, lamb, depth)
            
            # ground_truth_tree = tl.apply_missing_data(ground_truth_tree, p_miss)
            
            triplets = tl.find_recon_triplets(ground_truth_tree)
            recon_tree = tl.build_tree_from_triplet_partition(ground_truth_tree, triplets)
            
            acc = tl.calculate_triplets_correct(ground_truth_tree, recon_tree)
            
            if acc >= acc_threshold:
                success_count += 1
                
        prob = success_count / num_simulations_fig2
        results_fig2.append({
            "p_miss": f"$p_{{miss}} = {p_miss}$", 
            "k_cand": k_cand, 
            "Success_Probability": prob
        })

df_fig2 = pd.DataFrame(results_fig2)
df_fig2.to_csv("fig2_success_probability.csv", index=False)
print("Data saved to fig2_success_probability.csv")

# =========================================================
# =========================================================
print("\n========== Generating Data for Figure 1 (Required Capacity K) ==========")
def get_empirical_k_for_target(target_acc, p_miss_val=0.0):
    k_test_range = [10, 30, 50, 100, 200]
    acc_results = []
    for k in k_test_range:
        acc_vals = []
        for _ in range(5):
            gt_tree = tl.complete_binary_tree_sim(k, q_dist, lamb, depth)
            # gt_tree = tl.apply_missing_data(gt_tree, p_miss_val)
            trips = tl.find_recon_triplets(gt_tree)
            r_tree = tl.build_tree_from_triplet_partition(gt_tree, trips)
            acc_vals.append(tl.calculate_triplets_correct(gt_tree, r_tree))
        acc_results.append(np.mean(acc_vals))
    
    interp_func = interp1d(acc_results, k_test_range, kind='linear', fill_value="extrapolate")
    return float(interp_func(target_acc))

results_fig1 = []

for batch in tqdm(range(num_batches_fig1), desc="Running Batches for 95% CI"):
    for t_acc in target_acc_list:
        sim_k = get_empirical_k_for_target(t_acc, p_miss_val=0.0)
        results_fig1.append({
            "Target_Accuracy": t_acc,
            "Required_k": sim_k,
            "Type": "Simulated (MAX-Cut)"
        })

df_fig1 = pd.DataFrame(results_fig1)
df_fig1.to_csv("fig1_required_capacity.csv", index=False)
print("Data saved to fig1_required_capacity.csv")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# =========================================================
df_fig1 = pd.read_csv(PROJECT_ROOT / "results" / "fig1_required_capacity.csv")
df_fig2 = pd.read_csv(PROJECT_ROOT / "results" / "fig2_success_probability.csv")

# =========================================================
# =========================================================
depth = 8
def theoretical_k_bound(acc):
    """
    """
    return max(5, 8 * np.log(2**depth) / ((1.01 - acc)**1.5))

target_acc_list = sorted(df_fig1["Target_Accuracy"].unique())

# =========================================================
# =========================================================
sns.set_theme(style="whitegrid", rc={"axes.edgecolor": "black", "grid.color": "lightgrey"})
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ----------------- Panel A: Theoretical vs Simulated K -----------------
ax0 = axes[0]

sns.lineplot(
    data=df_fig1, 
    x="Target_Accuracy", 
    y="Required_k", 
    color="#FF8C00",
    marker="o",
    errorbar=("ci", 95),
    label="Simulated (MAX-Cut)",
    ax=ax0
)

theo_k_vals = [theoretical_k_bound(acc) for acc in target_acc_list]
ax0.plot(target_acc_list, theo_k_vals, color="#4682B4", linestyle="--", linewidth=2.5, label="Theoretical Limit (OTO)")

ax0.set_title("Validation of Required Recording Capacity", fontsize=14, fontweight='bold', pad=15)
ax0.set_xlabel(r"Target Accuracy ($Acc_{target}$)", fontsize=12)
ax0.set_ylabel(r"Required Capacity ($k_{req}$)", fontsize=12)
ax0.legend(loc="upper left", frameon=True, fontsize=10)

# ----------------- Panel B: Success Probability vs k -----------------
ax1 = axes[1]

sns.lineplot(
    data=df_fig2, 
    x="k_cand", 
    y="Success_Probability", 
    hue="p_miss", 
    palette="viridis",
    marker="s", 
    linewidth=2,
    ax=ax1
)

ax1.axhline(y=0.9, color='red', linestyle=':', linewidth=1.5, label="90% Reliability")

ax1.set_title("Global Tree Reconstruction Reliability", fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel(r"Number of Target Sites ($k_{cand}$)", fontsize=12)
ax1.set_ylabel(r"Success Probability ($Acc \geq 0.9$)", fontsize=12)
ax1.set_ylim(-0.05, 1.05)
ax1.legend(title="Missing Rate", loc="lower right", frameon=True, fontsize=10)

fig.text(0.04, 0.95, "a", fontsize=16, fontweight="bold")
fig.text(0.50, 0.95, "b", fontsize=16, fontweight="bold")

plt.tight_layout(rect=[0.05, 0, 1, 1])
plt.savefig(PROJECT_ROOT / "figures" / "capacity-reliability.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("Plots saved successfully as figures/capacity-reliability.pdf")